In [30]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [36]:
class FeedForward(nn.Module):

    def __init__(self, embed_size):
        super().__init__()
        self.linear1 = nn.Linear(embed_size, embed_size * 4)
        self.act = nn.SiLU()
        self.linear2 = nn.Linear(embed_size * 4, embed_size)

    def forward(self, x):
        x = self.act(self.linear1(x)) # (batch_size, num_patches, embed_size) -> (batch_size, num_patches, embed_size * 4)
        x = self.linear2(x) # (batch_size, num_patches, embed_size * 4) -> (batch_size, num_patches, embed_size)
        return x

class TransformerBlock(nn.Module):

    def __init__(self, embed_size, num_head):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_size)
        self.attn = nn.MultiheadAttention(embed_size, num_head, batch_first = True)
        self.norm2 = nn.LayerNorm(embed_size)
        self.ff = FeedForward(embed_size)
    
    def forward(self, x):
        norm_x = self.norm1(x) # (batch_size, num_patches, embed_size) -> (batch_size, num_patches, embed_size)
        attn_x = self.attn(key = norm_x, query = norm_x, value = norm_x)[0] 
        x = x + attn_x # (batch_size, num_patches, embed_size) -> (batch_size, num_patches, embed_size)
        x = self.ff(self.norm2(x)) + x # (batch_size, num_patches, embed_size) -> (batch_size, num_patches, embed_size)
        return x

class MLPClassificationHead(nn.Module):

    def __init__(self, embed_size, hidden_layer, classes_num):
        super().__init__()
        self.linear1 = nn.Linear(embed_size, hidden_layer)
        self.silu = nn.SiLU()
        self.linear2  = nn.Linear(hidden_layer, classes_num)
    
    def forward(self, x):
        x = self.linear1(x) # (batch_size, embed_size) -> (batch_size, hidden_layer)
        x = self.linear2(self.silu(x)) # (batch_size, hidden_layer) -> (batch_size, classes_num)
        return x


class ViT(nn.Module):

    def __init__(self, batch_size, channel_num, image_height, image_width,  patch_size, embed_size, blocks_num, classes_num, hidden_layer, num_head):
        super().__init__()
        p_height = image_height // patch_size
        p_width = image_width // patch_size
        patches_num = p_height * p_width
        self.patches_num = patches_num
        self.batch_size = batch_size
        self.patch_size = patch_size
        self.p_height = p_height
        self.p_width = p_width
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_size))
        self.pos_enc = nn.Embedding(patches_num + 1, embed_size)
        self.linear1 = nn.Linear(channel_num * patch_size * patch_size, embed_size)
        self.transformer_blocks = nn.Sequential(*[TransformerBlock(embed_size, num_head) for _ in range(blocks_num)])
        self.mlp_head = MLPClassificationHead(embed_size, hidden_layer, classes_num)
    
    def forward(self, x): # x - (batch_size, channels, height, width)
        B, C, H, W = x.shape
        x = x.reshape(B, C, self.p_height, self.patch_size, self.p_width, self.patch_size) 
        x = x.permute(0, 2, 4, 3, 5, 1)
        x = x.reshape(B, self.patches_num, (self.patch_size ** 2) * C)
        x = self.linear1(x) # (B, patches_num, patch_size ** 2 * C) -> (B, patches_num, embed_size)
        x = torch.cat([self.cls_token.expand(B, -1, -1), x], dim = 1)
        x = x + self.pos_enc(torch.arange(self.patches_num + 1).to(device)) # (B, patches_num, embed_size) + (patches_num, embed_size) -> (B, patches_num, embed_size)
        x = self.transformer_blocks(x)
        cls_out = x[:, 0] # (B, embed_size)
        logits = self.mlp_head(cls_out)
        return logits

In [43]:
batch_size = 32
channel_num = 3
image_height = 64
image_width = 64
patch_size = 8
embed_size = 64
blocks_num = 4
classes_num = 10
hidden_layer = 128
num_head = 4

In [44]:
model = ViT(batch_size, channel_num, image_height, image_width, patch_size, embed_size, blocks_num, classes_num, hidden_layer, num_head)

In [45]:
sum(p.numel() for p in model.parameters())

226122

In [40]:
random_batch = torch.randn((batch_size, channel_num, image_height, image_width))

In [41]:
model(random_batch)

tensor([[-7.3585e-04,  5.9273e-01, -1.6602e-01,  6.6477e-02,  6.1974e-01,
          1.4777e-01,  6.5533e-01, -1.7277e-01,  1.6059e-01, -9.4539e-01],
        [ 2.8568e-02,  5.6663e-01, -1.6403e-01,  7.3555e-02,  6.3064e-01,
          1.5977e-01,  6.4157e-01, -1.7291e-01,  1.8619e-01, -9.3568e-01],
        [ 2.2019e-02,  5.8573e-01, -1.4196e-01,  4.5810e-02,  6.1863e-01,
          1.7846e-01,  6.3781e-01, -1.7011e-01,  1.4537e-01, -9.3489e-01],
        [ 9.3843e-03,  5.9993e-01, -1.5337e-01,  6.4094e-02,  6.2287e-01,
          1.8123e-01,  6.4010e-01, -1.9704e-01,  1.4469e-01, -9.4999e-01],
        [ 2.9923e-02,  5.8135e-01, -1.6131e-01,  6.2376e-02,  6.3305e-01,
          1.6656e-01,  6.6173e-01, -1.6206e-01,  1.7067e-01, -9.5020e-01],
        [ 2.4259e-02,  5.8065e-01, -1.4439e-01,  7.3142e-02,  6.0061e-01,
          1.6323e-01,  6.6087e-01, -1.7427e-01,  1.6966e-01, -9.4046e-01],
        [ 3.4280e-02,  5.7244e-01, -1.5312e-01,  5.6566e-02,  6.1860e-01,
          1.6565e-01,  6.4341e-0